# Generate the synthetic NovaCart support ticket dataset

Generates simulated Zendesk-style tickets and conversation threads for the
26-category taxonomy defined in `src/taxonomy.yaml`. No LLM calls are used —
everything is templated text + [Faker](https://faker.readthedocs.io/), with
built-in messiness (typos, conversation drift, ambiguous categories) so the
classifier comparison in the next notebook has something realistic to chew on.

Outputs `outputs/tickets.csv` and `outputs/conversations.csv`.

In [1]:
import sys
sys.path.append("../src")

import pandas as pd

from taxonomy import load_taxonomy
from data_generation import generate_dataset, save_dataset

pd.set_option("display.max_colwidth", 120)

In [2]:
taxonomy = load_taxonomy()
print(f"Taxonomy v{taxonomy.taxonomy_version}: {len(taxonomy.categories)} categories")
for group in sorted({c.group for c in taxonomy.categories}):
    ids = [c.category_id for c in taxonomy.categories if c.group == group]
    print(f"  {group}: {len(ids)} -> {ids}")

Taxonomy v1: 26 categories
  Account: 3 -> ['ACC-001', 'ACC-002', 'ACC-003']
  General: 4 -> ['GEN-001', 'GEN-002', 'GEN-003', 'GEN-OTHER']
  Loyalty: 1 -> ['LOY-001']
  Order & shipping: 5 -> ['ORD-001', 'ORD-002', 'ORD-003', 'ORD-004', 'ORD-005']
  Payments & billing: 4 -> ['PAY-001', 'PAY-002', 'PAY-003', 'PAY-004']
  Product: 5 -> ['PRD-001', 'PRD-002', 'PRD-003', 'PRD-004', 'PRD-005']
  Returns & refunds: 4 -> ['RET-001', 'RET-002', 'RET-003', 'RET-004']


In [3]:
tickets_df, conversations_df = generate_dataset(
    n_tickets=1200,
    taxonomy=taxonomy,
    seed=42,
    drift_rate=0.12,
    ambiguous_rate=0.08,
)

print(f"{len(tickets_df)} tickets, {len(conversations_df)} conversation messages")
tickets_df.head()

1200 tickets, 7146 conversation messages


,id,subject,description,status,priority,type,via_channel,tags,requester_id,assignee_id,group_id,created_at,updated_at,true_category_id_triage,true_category_id_final,taxonomy_version,is_drift_case,is_ambiguous_case,ambiguous_with_category_id,is_injected_new_category
0,10000,Package hasn't arrived yet,"Hi, my delivry is running late, I ordered the yoga mat over a week ago.",solved,low,task,web_widget,web,52619,903,1,2025-06-26T00:48:15,2025-06-26T14:45:15,ORD-002,ORD-002,1,False,False,NaN,False
1,10001,Package hasn't arrived yet,Order #496922 was supposed to arrive on Jul 09 and it's stil not here. What's going on?,solved,normal,incident,web_widget,,51083,925,1,2025-07-12T19:37:34,2025-07-13T01:53:34,ORD-002,ORD-002,1,False,False,NaN,False
2,10002,When will this be back in stock?,"Hi, the leather wallet shows as out of stock, do you know when it'll be available again?",solved,normal,incident,web_widget,,50669,914,1,2025-08-24T05:32:40,2025-08-24T22:49:40,PRD-004,PRD-004,1,False,False,NaN,False
3,10003,Package hasn't arrived yet,"Hi, my delivery is running late, I ordered the phone case over a week ago.",solved,normal,question,web_widget,ecommerce,53095,901,2,2025-06-14T21:53:40,2025-06-15T16:24:40,ORD-002,GEN-003,1,True,False,NaN,False
4,10004,Address correction for #160738,"I moved recently and my order #160738 is still set to go to my old address, can this be fixed?",solved,low,incident,web_widget,,50458,911,2,2025-08-15T16:48:51,2025-08-16T13:42:51,ORD-005,ORD-005,1,False,False,NaN,False


## Sanity checks

- Category distribution is imbalanced by design, but every active category should appear.
- Drift cases: the conversation reveals a different underlying issue than the subject/opening message implied — final classification should differ from a triage-only read.
- Ambiguous cases: the opener blends language from two confusable categories.

In [4]:
counts = tickets_df["true_category_id_final"].value_counts()
missing = set(taxonomy.active_ids()) - set(counts.index)
print(f"Categories with zero tickets: {missing or 'none'}")
counts

Categories with zero tickets: none


true_category_id_final
PRD-001      138
ORD-002      129
RET-002      129
ORD-001      106
PAY-002       49
GEN-OTHER     42
PRD-004       41
ACC-001       38
ORD-003       38
PAY-004       38
ACC-002       36
PRD-002       36
ORD-005       35
RET-004       35
RET-003       35
PAY-001       35
GEN-001       35
PRD-003       33
PAY-003       33
RET-001       30
PRD-005       27
ORD-004       22
GEN-002       16
LOY-001       16
GEN-003       15
ACC-003       13
Name: count, dtype: int64

In [5]:
print("Drift cases (triage category != final category):")
drift = tickets_df[tickets_df["is_drift_case"]]
print(f"  {len(drift)} tickets ({len(drift) / len(tickets_df):.1%})")
drift[["subject", "true_category_id_triage", "true_category_id_final"]].head()

Drift cases (triage category != final category):
  146 tickets (12.2%)


,subject,true_category_id_triage,true_category_id_final
3,Package hasn't arrived yet,ORD-002,GEN-003
14,Delivery delay on #821024,ORD-002,RET-004
23,Still waiting on refund for #385470,RET-002,RET-004
35,Order #763829 missing an item,ORD-004,GEN-002
55,Return process question,RET-001,PRD-002


In [6]:
print("Ambiguous cases (opener blends two confusable categories):")
amb = tickets_df[tickets_df["is_ambiguous_case"]]
print(f"  {len(amb)} tickets ({len(amb) / len(tickets_df):.1%})")
amb[["description", "true_category_id_final", "ambiguous_with_category_id"]].head()

Ambiguous cases (opener blends two confusable categories):
  45 tickets (3.8%)


,description,true_category_id_final,ambiguous_with_category_id
11,"Hi, I was only refunded part of my order, can you check what happened? Hi, the amount charged to my card doesn't mat...",RET-003,PAY-002
47,"I got refunded for order #216242 but the amount is way less than what I paid. Hi, the amount charged to my card does...",RET-003,PAY-002
54,"I ordered the denim jacket but recieved a completely different item. Hi, my order arrived and the denim jacket is br...",PRD-002,PRD-001
57,"Hi, when will I get my money back for the item I sent back? I was charged twice for order #933510, can you refund th...",RET-002,PAY-002
73,"Hi, the courier says my package for order #825916 was delivered but it's nowhere to be found. Hi, when will I get my...",ORD-003,RET-002


In [7]:
# Example full conversation thread for one ticket
sample_id = tickets_df.iloc[0]["id"]
conversations_df[conversations_df["ticket_id"] == sample_id][["author_id", "body", "created_at"]]

,author_id,body,created_at
0,52619,"Hi, my delivry is running late, I ordered the yoga mat over a week ago.",2025-06-26T00:48:15
1,903,"I'm sorry for the trouble, checking your account now.",2025-06-26T03:16:15
2,52619,Do you have an updated delivery estimate?,2025-06-26T08:05:15
3,903,"I'm sorry for the trouble, checking your account now.",2025-06-26T08:18:15
4,52619,Any news?,2025-06-26T11:26:15
5,903,"This has been taken care of, apologies again for the inconvenience.",2025-06-26T13:03:15
6,52619,"Perfect, thank you.",2025-06-26T14:45:15


## Save to outputs/

In [8]:
save_dataset(tickets_df, conversations_df, out_dir="../outputs")
print("Saved outputs/tickets.csv and outputs/conversations.csv")

Saved outputs/tickets.csv and outputs/conversations.csv
